# Exploratory Data Analysis: Post-Harvest Food Loss Prediction

This notebook performs exploratory data analysis on the EuroCrop agricultural logistics dataset located at `data/EuroCrop_agricultural_logistics_dataset.csv`. 

### Core Analytical Insights:
1. **Severe Data Corruption**: The raw numeric variables are scaled exponentially (up to $10^{308}$) and contain a high number of infinite values. This causes standard estimators to overflow to NaN.
2. **Log-Reconstruction Strategy**: Applying a natural log transformation ($x = \ln(y)$) successfully reverses the exponential scaling, recovering standard agricultural ranges (e.g. temperatures of $5\text{--}18^\circ\text{C}$ and humidity of $60\text{--}90\%$).
3. **Null Prediction Result**: The target variable `Spoilage_Risk` has near-zero correlation ($< 0.01$) with all individual input features. Machine learning classifiers are expected to perform at the dummy baseline (~33.3% accuracy).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_raw_data, split_data, create_target_labels

sns.set_theme(style="whitegrid")

## 1. Load and Inspect Raw Dataset

In [ ]:
df = load_raw_data("data/EuroCrop_agricultural_logistics_dataset.csv")
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Missing and Infinite Values Analysis

We calculate the percentage of missing (NaN) and infinite values per column.

In [ ]:
missing_df = pd.read_csv('reports/missing_value_analysis.csv')
print("Features with missing or infinite values:")
missing_df[missing_df['Corrupt_Percentage'] > 0].sort_values(by='Corrupt_Percentage', ascending=False)

### Visualizing Missing/Infinite Rates

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=missing_df[missing_df['Corrupt_Percentage'] > 0].sort_values(by='Corrupt_Percentage', ascending=False),
            x='Corrupt_Percentage', y='Feature', palette='viridis', hue='Feature', legend=False)
plt.title('Percentage of Missing/Infinite Values by Feature')
plt.xlabel('Percentage (%)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## 3. Class Distribution of the Target

We create the class labels based *only* on the training target percentiles (33rd and 66th percentiles) to prevent data leakage.

In [ ]:
X_train_raw, X_test_raw, y_train_cont, y_test_cont = split_data(df)
y_train_cat, y_test_cat, thresholds = create_target_labels(y_train_cont, y_test_cont)

print(f"Calculated Thresholds:\n- 33rd Percentile (Low/Med boundary): {thresholds['33rd_percentile']:.6f}\n- 66th Percentile (Med/High boundary): {thresholds['66th_percentile']:.6f}")
print("\nTrain class distribution:")
print(y_train_cat.value_counts())
print("\nTest class distribution:")
print(y_test_cat.value_counts())

### Plotting Class Distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=y_train_cat, hue=y_train_cat, order=['Low Risk', 'Medium Risk', 'High Risk'], palette='coolwarm', legend=False)
plt.title('Risk Category Distribution in Training Set')
plt.xlabel('Risk Category')
plt.ylabel('Count')
plt.show()

## 4. Log Reconstruction of Skewed Features

Here we show the distributions of key variables before and after the log-reconstruction transformation. This demonstrates how exponentiated values are returned to their natural scales.

In [ ]:
features_to_plot = ['Storage_Temperature', 'Storage_Humidity', 'Warehouse_Storage_Time']
fig, axes = plt.subplots(3, 2, figsize=(14, 15))

for i, col in enumerate(features_to_plot):
    raw_data = df[col].replace([np.inf, -np.inf], np.nan).dropna()
    log_data = np.log1p(raw_data)
    
    # Histograms
    sns.histplot(raw_data, bins=30, ax=axes[i, 0], color='salmon', kde=False)
    axes[i, 0].set_title(f"Raw {col} (Exponentiated Scale)")
    axes[i, 0].set_xlabel('Raw Value')
    
    sns.histplot(log_data, bins=30, ax=axes[i, 1], color='skyblue', kde=True)
    axes[i, 1].set_title(f"Log-Reconstructed {col} (Physical Scale)")
    axes[i, 1].set_xlabel('Log-Transformed Value')

plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

We plot the correlation heatmap of the preprocessed variables with `Spoilage_Risk` to verify if any feature has a direct relationship with the target.

In [ ]:
cols_to_drop = ['Inventory_Levels', 'Vehicle_Load_Capacity', 'Crop_Yield', 'Station_Capacity', 'Operational_Cost', 'Energy_Consumption', 'Efficiency_Ratio', 'Unnamed: 0', 'Harvest_Date']
clean_df = df.drop(columns=[col for col in cols_to_drop if col in df.columns]).copy()
clean_numeric = clean_df.select_dtypes(include=[np.number]).columns.tolist()

clean_df[clean_numeric] = clean_df[clean_numeric].replace([np.inf, -np.inf], np.nan)
for col in clean_numeric:
    clean_df[col] = np.log1p(clean_df[col]).fillna(clean_df[col].median())

corr_matrix = clean_df.corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1.0, vmax=1.0)
plt.title('Correlation Matrix (Pearson)')
plt.tight_layout()
plt.show()

In [ ]:
print("Top correlations with Spoilage_Risk:")
corr_matrix['Spoilage_Risk'].drop('Spoilage_Risk').sort_values(ascending=False)